In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

# 데이터 준비
wine = pd.read_csv('https://bit.ly/wine_csv_data')
X = wine[['alcohol', 'sugar', 'pH']].to_numpy()
y = wine['class'].to_numpy()

# train/test 분할
train_input, test_input, train_target, test_target = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# train set에서 다시 validation set으로 분할
sub_train_input, val_input, sub_train_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42, stratify=train_target
)

# 랜덤 포레스트 모델 : 순서대로 랜덤으로 생성할 결정트리의 개수 / 각 트리들의 최대 깊이 / 트리에서 노드 분할 시 고려할 특성 개수 이다.
rf_params = [
    {"n_estimators": 100, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 5,    "max_features": "log2"},
]

# 엑스트라 트리 모델 : 랜덤 포레스트 모델과 같다.
et_params = [
    {"n_estimators": 100, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 300, "max_depth": 5,    "max_features": "log2"},
]

# 그래디언트 부스팅 모델 : 러닝 레이트는 학습률인데, 낮으면 낮을수록 더 조심스럽게 모델이 학습한다. 서브 샘플은 추출한 데이터들에서 일부만 사용해 랜덤성을 부여한다.
gb_params = [
    {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 2, "subsample": 1.0},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "subsample": 1.0},
    {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 3, "subsample": 0.8},
]

# 히스토그램 그래디언트 부스팅 모델 : min_samples_leaf는 리프에 필요한 최소 표본 수이다.
hgb_params = [
    {"learning_rate": 0.1, "max_depth": None, "min_samples_leaf": 20},
    {"learning_rate": 0.05, "max_depth": 6,   "min_samples_leaf": 20},
    {"learning_rate": 0.05, "max_depth": 6,   "min_samples_leaf": 30},
]


# 아까 정의한 모델 파라미터들에서 하나씩 꺼내서 모델을 훈련하고, 결과값 저장
# 더 결과가 좋다면 결과값 갱신하여 출력
def try_params(model_name, ModelClass, param_list):
    best_acc = -1.0
    best_params = None
    best_model = None

    print(f"\n[{model_name}] 파라미터 테스트 시작")
    for i, params in enumerate(param_list, start=1):
        model = ModelClass(random_state=42, n_jobs=-1) if "n_jobs" in ModelClass.__init__.__code__.co_varnames else ModelClass(random_state=42)
        for k, v in params.items():
            try:
                setattr(model, k, v)
            except Exception:
                pass

        # 학습
        model.fit(sub_train_input, sub_train_target)
        # 검증 정확도
        val_pred = model.predict(val_input)
        acc = accuracy_score(val_target, val_pred)
        print(f"  - 조합 {i}: params={params} -> 검증 정확도={acc:.4f}")

        # 최고 성능 갱신
        if acc > best_acc:
            best_acc = acc
            best_params = params
            best_model = model

    print(f"[{model_name}] 최고 검증 정확도={best_acc:.4f}, 최고 파라미터={best_params}")
    return best_model, best_acc, best_params

# 4개 모델들을 각각 테스트
rf_best, rf_acc, rf_bp = try_params("RandomForest", RandomForestClassifier, rf_params)
et_best, et_acc, et_bp = try_params("ExtraTrees", ExtraTreesClassifier, et_params)
gb_best, gb_acc, gb_bp = try_params("GradientBoosting", GradientBoostingClassifier, gb_params)
hgb_best, hgb_acc, hgb_bp = try_params("HistGradientBoosting", HistGradientBoostingClassifier, hgb_params)

# 어떤 모델이 제일 좋았는지 선택
candidates = [
    ("RandomForest", rf_best, rf_acc, rf_bp),
    ("ExtraTrees", et_best, et_acc, et_bp),
    ("GradientBoosting", gb_best, gb_acc, gb_bp),
    ("HistGradientBoosting", hgb_best, hgb_acc, hgb_bp),
]
top_name, top_model, top_val_acc, top_params = sorted(candidates, key=lambda x: x[2], reverse=True)[0]
print(f"\n[검증 1등] {top_name} (검증 정확도={top_val_acc:.4f}, params={top_params})")

# 제일 좋았던 모델을 학습데이터 전체로 다시 학습해서 테스트 점수 확인
top_model.fit(train_input, train_target)
test_pred = top_model.predict(test_input)
test_acc = accuracy_score(test_target, test_pred)
print(f"\n[{top_name}] 테스트 정확도 = {test_acc:.4f}\n")
print("분류 리포트(테스트셋):\n", classification_report(test_target, test_pred, digits=4))




[RandomForest] 파라미터 테스트 시작
  - 조합 1: params={'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt'} -> 검증 정확도=0.8990
  - 조합 2: params={'n_estimators': 300, 'max_depth': None, 'max_features': 'sqrt'} -> 검증 정확도=0.8981
  - 조합 3: params={'n_estimators': 300, 'max_depth': 5, 'max_features': 'log2'} -> 검증 정확도=0.8702
[RandomForest] 최고 검증 정확도=0.8990, 최고 파라미터={'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt'}

[ExtraTrees] 파라미터 테스트 시작
  - 조합 1: params={'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt'} -> 검증 정확도=0.8962
  - 조합 2: params={'n_estimators': 300, 'max_depth': None, 'max_features': 'sqrt'} -> 검증 정확도=0.8962
  - 조합 3: params={'n_estimators': 300, 'max_depth': 5, 'max_features': 'log2'} -> 검증 정확도=0.7548
[ExtraTrees] 최고 검증 정확도=0.8962, 최고 파라미터={'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt'}

[GradientBoosting] 파라미터 테스트 시작
  - 조합 1: params={'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'subsample': 1.0} -> 검증 정확도=0.8808
  - 